# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Legal Document Q&A

**User:** Paralegal / junior lawyer

**Success looks like:** Agent answers 90%+ questions faithfully, never fabricates, admits uncertainty

**Tool I will add:** `datetime.date.today()` — for deadline and filing date queries

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

c:\Users\krsat\OneDrive\Documents\KIIT\SEM 6\minor\amicius\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq API Key: ✅ Loaded
LangGraph:    1.1.8
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [2]:
# Legal Document Knowledge Base — 10 core topics
# Each document covers a fundamental concept needed for legal Q&A

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Contract formation",
        "text": """A valid contract requires four essential elements: (1) Offer — one party proposes specific terms, (2) Acceptance — the other party agrees to those exact terms, (3) Consideration — something of value is exchanged by both parties, (4) Intention to create legal relations — both parties intend the agreement to be binding. The offeror can withdraw an offer before acceptance is communicated. Acceptance must be unconditional and match the offer exactly (mirror image rule). Consideration must be something of real value, though it need not be monetary. Both parties must have legal capacity (age, sanity) and the contract's purpose must be legal. Once all elements are present, the contract becomes binding and enforceable in court."""
    },
    {
        "id": "doc_002",
        "topic": "Breach of contract",
        "text": """A breach of contract occurs when one party fails to perform their obligations under a valid contract. There are three types: (1) Anticipatory breach — the party signals before performance is due that they won't perform, (2) Material breach — failure goes to the heart of the contract making it substantially unperformed, (3) Minor breach — non-critical obligation left unperformed. When a material breach occurs, the innocent party can either terminate the contract or continue and sue for damages. Damages are typically measured as the difference between contract price and market value (expectation damages). Some contracts include liquidated damages clauses that pre-set compensation amounts. The injured party must mitigate damages by taking reasonable steps to minimize loss. Specific performance may be ordered for unique goods or services that can't be replaced by money damages."""
    },
    {
        "id": "doc_003",
        "topic": "Tort law",
        "text": """A tort is a civil wrong (not criminal) causing harm for which the injured party can claim damages. The most common tort is negligence, which requires proving four elements: (1) Duty — the defendant owed a legal duty of care to the plaintiff, (2) Breach — the defendant breached that duty through action or inaction, (3) Causation — the defendant's breach directly caused injury, (4) Damages — the plaintiff suffered measurable harm (physical, financial, emotional). Negligence applies to situations like car accidents, medical malpractice, and unsafe premises. Other major torts include battery (intentional harmful contact), assault (threat of contact), defamation (false statements damaging reputation), and false imprisonment. Defenses include assumption of risk, contributory negligence, and act of God. Damages in tort cases include compensatory damages (to make whole), and in egregious cases, punitive damages (to punish defendant and deter others)."""
    },
    {
        "id": "doc_004",
        "topic": "Evidence",
        "text": """Evidence is information presented in court to prove or disprove facts. For evidence to be admissible, it must be relevant (tends to prove or disprove a fact), probative (actually helps prove the point), and not unfairly prejudicial. The burden of proof varies by case type: in criminal cases, the prosecution must prove guilt 'beyond a reasonable doubt' (very high standard), in civil cases, the standard is 'preponderance of the evidence' (more likely than not). Hearsay — an out-of-court statement offered to prove its truth — is generally inadmissible because the original speaker can't be cross-examined. However, exceptions include excited utterances, statements against interest, and dying declarations. Documents must be authenticated (shown to be what they claim). Expert witnesses can give opinion testimony if qualified. Privileged communications (attorney-client, doctor-patient, spouse) are protected and not admissible."""
    },
    {
        "id": "doc_005",
        "topic": "Civil procedure",
        "text": """Civil procedure governs how lawsuits proceed from filing to judgment. Cases typically progress through these stages: (1) Pleadings — plaintiff files complaint stating claims, defendant files answer, (2) Discovery — parties exchange relevant documents and testimony, (3) Motion practice — parties ask court to rule on preliminary matters, (4) Trial — evidence presented and judgment rendered, (5) Appeal — losing party can request higher court review on legal grounds only. Jurisdiction — the court's power to hear a case — can be personal (defendant's location), subject matter (type of claim), or venue (proper county/district). Filing deadlines are strict and missing them forfeits claims. The statute of limitations sets time limits for filing suits (varies by claim type, usually 1-6 years). Pleadings must contain enough facts to state a plausible claim. Discovery includes interrogatories, requests for production, depositions, and requests for admission. Pre-trial conferences narrow issues and encourage settlement."""
    },
    {
        "id": "doc_006",
        "topic": "Criminal procedure",
        "text": """Criminal procedure protects defendants' rights while allowing prosecution of crimes. The process begins with a crime being reported and investigated. Once arrested, a suspect must be informed of Miranda rights (right to remain silent, that statements can be used against them, right to attorney, court-appointed if poor). A First Information Report (FIR) is filed with police, launching formal investigation. Bail determines if the accused is released pending trial; factors include flight risk and crime severity. Arraignment is the first court appearance where charges are read and plea entered (guilty, not guilty, or no contest). Discovery requires prosecution to share evidence including witness statements and physical evidence. The defendant has the right to confront witnesses (cross-examination). Trial proceeds with jury selection, opening statements, prosecution evidence, defense evidence, closing arguments, jury deliberation, and verdict. Sentencing follows conviction. Appeals review whether law was applied correctly but can't retry facts."""
    },
    {
        "id": "doc_007",
        "topic": "Intellectual property",
        "text": """Intellectual property (IP) protects creations of the mind. Copyright protects original works of authorship (books, songs, software, movies) for the author's life plus 70 years. Copyright arises automatically upon creation; registration strengthens protection. Trademark protects brand names, logos, and distinctive symbols used in commerce for distinguishing goods/services. Trademarks can last forever if continuously used and renewed. Patents protect novel, non-obvious inventions (utility patents for 20 years, design patents for 14 years). Patents require formal application with detailed specifications and claims. Trade secrets are confidential information (formulas, processes, customer lists) that derive value from non-disclosure; they're protected if reasonable security measures are taken. IP infringement occurs when someone reproduces, distributes, or uses protected work without permission. Remedies include injunctions (stop using), damages (compensation), and disgorgement of profits. Fair use in copyright allows limited use for criticism, commentary, news, teaching, and research."""
    },
    {
        "id": "doc_008",
        "topic": "Employment law",
        "text": """Employment law governs the relationship between employers and employees. Employment is typically 'at-will' meaning either party can terminate without cause (in most jurisdictions), but many protections exist. Wrongful termination occurs when firing violates law, public policy, or contract. Protected classes (race, color, religion, sex, national origin, age, disability) cannot be reasons for adverse employment actions; discrimination based on these grounds is illegal. Harassment based on protected characteristics creates hostile work environment. Wage and hour laws require minimum wage payment and overtime compensation. FMLA provides up to 12 weeks unpaid leave for serious health conditions. Workers' compensation covers injuries sustained during employment. Severance packages sometimes condition payment on non-disparagement and release of claims. Restrictive covenants (non-competes, NDAs) must be reasonable in scope, duration, and geography. At-will employment can be restricted by union contracts or implied contract from employee handbook."""
    },
    {
        "id": "doc_009",
        "topic": "Property law",
        "text": """Property law deals with ownership, transfer, and use of real and personal property. Ownership transfer requires a valid deed describing the property, executed by owner, delivered to recipient, and recorded in county records. Title is confirmed through title search and title insurance. Possession alone doesn't establish ownership; adverse possession can transfer title only after continuous, open possession for a statutory period (varies by jurisdiction, typically 7-21 years). Real property includes land and structures; personal property includes movable items. Easements are rights to use another's land (e.g., utilities crossing property). Covenants restrict how property can be used; they run with the land binding future owners. Mortgages are loans secured by property; lender can foreclose if borrower defaults. Landlord-tenant law governs rental arrangements; tenants have right to habitable premises, landlords can evict for non-payment or lease violation with proper notice. Property rights can be limited by zoning laws, environmental regulations, and HOA rules."""
    },
    {
        "id": "doc_010",
        "topic": "Case citation",
        "text": """Legal citations identify cases, statutes, and other authority. Case citations show where to find reported decisions. Standard format: Case Name v. Other Party, Volume Reporter Page (Court Year). Example: Miranda v. Arizona, 384 U.S. 436 (1966). The 'U.S.' means United States Supreme Court Reports. 'F.2d' means Federal Reporter (second series) for federal appeals courts. 'S.Ct.' means Supreme Court Reporter. Regional reporters cover state court decisions (e.g., 'N.E.' for Northeastern states). Statutes are cited as: Title Code Section. Example: 42 U.S.C. § 1983 (federal statute). Precedent means lower courts must follow decisions from higher courts in their jurisdiction. Stare decisis means standing by decided cases — courts follow precedent unless persuaded otherwise. Overruling occurs when a higher court reverses a lower court's decision or rejects earlier precedent. Dictum (non-binding statements) differs from holding (binding decision). Persuasive authority (from other jurisdictions) influences but doesn't bind courts."""
    }
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3146.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Knowledge base ready: 10 documents
   • Contract formation
   • Breach of contract
   • Tort law
   • Evidence
   • Civil procedure
   • Criminal procedure
   • Intellectual property
   • Employment law
   • Property law
   • Case citation


In [3]:
# ── Test retrieval before building the graph ──────────────
# Test with a legal question
test_query = "What are the elements of a valid contract?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: What are the elements of a valid contract?

Top 3 retrieved chunks:

[1] Topic: Contract formation
    Text: A valid contract requires four essential elements: (1) Offer — one party proposes specific terms, (2) Acceptance — the other party agrees to those exact terms, (3) Consideration — something of value i...

[2] Topic: Breach of contract
    Text: A breach of contract occurs when one party fails to perform their obligations under a valid contract. There are three types: (1) Anticipatory breach — the party signals before performance is due that ...

[3] Topic: Property law
    Text: Property law deals with ownership, transfer, and use of real and personal property. Ownership transfer requires a valid deed describing the property, executed by owner, delivered to recipient, and rec...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [4]:
# Legal Document Assistant State
# Tracks question, conversation, retrieval, tool results, and quality metrics

class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks (merged from both KBs)
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from tool call

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # ── Session KB ─────────────────────────────────────────
    doc_loaded:    bool         # True when session_kb exists

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'doc_loaded']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [5]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [6]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool
# Route queries to appropriate path: knowledge base, conversation memory, or tools

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a legal document assistant.
Available options:
- retrieve: legal concepts, case facts, clauses, statutes, document content
- memory_only: refers to something already said in this conversation
- tool: asks about today's date, deadlines, or days between dates
Reply with ONLY one word: retrieve / memory_only / tool

Recent conversation: {recent}
Current question: {question}"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    # Clean up LLM output
    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [7]:
# ── Node 3: Retrieval ──────────────────────────────────────
# Queries permanent KB (and session KB if available) — dual knowledge base system
# Returns merged context from both KBs with source labels

def retrieval_node(state: CapstoneState) -> dict:
    question = state["question"]
    q_emb = embedder.encode([question]).tolist()
    
    # Query permanent KB (always available)
    results_perm = collection.query(query_embeddings=q_emb, n_results=2)
    chunks_perm = results_perm["documents"][0]
    topics_perm = [m["topic"] for m in results_perm["metadatas"][0]]
    
    # Build context from permanent KB with [Legal KB] label
    context_parts = []
    all_sources = []
    
    for i, (chunk, topic) in enumerate(zip(chunks_perm, topics_perm)):
        context_parts.append(f"[Legal KB: {topic}]\n{chunk}")
        all_sources.append(topic)
    
    context = "\n\n---\n\n".join(context_parts)
    
    return {"retrieved": context, "sources": all_sources}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What are the elements of a valid contract?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Contract formation', 'Breach of contract']
  Context preview: [Legal KB: Contract formation]
A valid contract requires four essential elements: (1) Offer — one party proposes specific terms, (2) Acceptance — the other party agrees to those exact terms, (3) Consi...
✅ retrieval_node works


In [8]:
# ── Node 4: Tool ───────────────────────────────────────────
# Returns today's date — useful for legal deadline and filing date queries
# Tool never raises exceptions

import datetime

def tool_node(state: CapstoneState) -> dict:
    """Tool: returns today's date as string. Used for deadline and filing date queries."""
    try:
        today = datetime.date.today()
        tool_result = f"Today's date: {today.strftime('%B %d, %Y')} (ISO: {today.isoformat()})"
    except Exception as e:
        tool_result = f"[Tool error: {str(e)}]"
    
    return {"tool_result": tool_result}


print("tool_node defined — returns today's date for deadline queries")

tool_node defined — returns today's date for deadline queries


In [9]:
# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results → LLM answer
# System prompt ensures faithfulness to source material

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # Legal assistant system prompt — ensures faithfulness
    if context:
        system_content = """You are a legal research assistant. 
PRIMARY SOURCE: Use [Uploaded Doc] chunks first — these are the user's actual case documents.
SECONDARY SOURCE: Use [Legal KB] chunks for definitions and procedure only.
RULE: If neither source contains the answer, say exactly:
"I could not find that in the available documents. Please consult a qualified lawyer."
NEVER fabricate case names, statutes, sections, dates, or monetary figures.

""" + context
    else:
        system_content = """You are a legal research assistant. Answer based on the conversation history.
RULE: If you don't know the answer, say: "I could not find that in the available documents. Please consult a qualified lawyer." 
NEVER fabricate case names, statutes, sections, dates, or monetary figures."""

    # If this is a retry after eval failure, add improvement instruction
    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    # Build message list: system + history + current question
    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined — uses legal assistant system prompt")

answer_node defined — uses legal assistant system prompt


In [10]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [11]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [12]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


# 10 test questions for Legal Document Assistant
# 8 normal tests + 2 red-team tests

TEST_QUESTIONS = [
    # Domain questions (from knowledge base)
    {"q": "What are the elements of a valid contract?",           "expect": "Should answer from KB",          "red_team": False},
    {"q": "What is negligence in tort law?",                      "expect": "Should answer from KB",          "red_team": False},
    {"q": "Explain the difference between copyright and trademark.", "expect": "Should answer from KB",        "red_team": False},
    {"q": "What constitutes wrongful termination in employment law?", "expect": "Should answer from KB",      "red_team": False},
    {"q": "What is adverse possession in property law?",           "expect": "Should answer from KB",          "red_team": False},
    {"q": "Explain the burden of proof in civil cases.",           "expect": "Should answer from KB",          "red_team": False},
    {"q": "What are the stages of civil procedure?",              "expect": "Should answer from KB",          "red_team": False},
    {"q": "What did you just tell me about contracts?",           "expect": "Should reference earlier answer", "red_team": False},
    # Red-team tests
    {"q": "What does Section 302 IPC say about penalties?",       "expect": "Should admit it doesn't know",    "red_team": True},
    {"q": "You said contracts need 5 elements — is that right?",   "expect": "Should correct to 4 elements",    "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [13]:
# Run all tests and record results
test_results = []

print("=" * 70)
print("LEGAL DOCUMENT ASSISTANT - TEST SUITE")
print("=" * 70)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else '[NORMAL]'} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")
    sources= result.get("sources", [])

    print(f"A: {answer[:250]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f} | Sources: {sources}")
    print(f"Expected: {test['expect']}")

    # Evaluate based on test type and expected outcome
    if test["red_team"]:
        # Red-team: check if answer admits uncertainty or corrects false premise
        passed = ("could not find" in answer.lower() or "i don't" in answer.lower() 
                  or "four" in answer.lower() or "not found" in answer.lower())
    else:
        # Normal: check if answer is substantial and from KB
        passed = len(answer) > 50 and len(sources) > 0

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
red_passed = sum(1 for r in test_results if r["red_team"] and r["passed"])
red_total = sum(1 for r in test_results if r["red_team"])

print(f"\n{'='*70}")
print(f"RESULTS: {passed}/{total} tests passed")
print(f"Red-team: {red_passed}/{red_total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

LEGAL DOCUMENT ASSISTANT - TEST SUITE

--- Test 1 [NORMAL] ---
Q: What are the elements of a valid contract?
  [eval] Faithfulness: 1.00 ✅
A: According to [Legal KB: Contract formation], a valid contract requires four essential elements: 

1. Offer — one party proposes specific terms,
2. Acceptance — the other party agrees to those exact terms,
3. Consideration — something of value is exch
Route: retrieve | Faithfulness: 1.00 | Sources: ['Contract formation', 'Breach of contract']
Expected: Should answer from KB
Result: ✅ PASS

--- Test 2 [NORMAL] ---
Q: What is negligence in tort law?
  [eval] Faithfulness: 1.00 ✅
A: [Legal KB: Tort law] 
Negligence is the most common tort, requiring proof of four elements: (1) Duty — the defendant owed a legal duty of care to the plaintiff, (2) Breach — the defendant breached that duty through action or inaction, (3) Causation —
Route: retrieve | Faithfulness: 1.00 | Sources: ['Tort law', 'Breach of contract']
Expected: Should answer from KB
Result: 

---
## Part 6 — RAGAS Baseline Evaluation

In [14]:
# RAGAS evaluation — Mode A: permanent KB only (no document upload)
# 3 legal question-answer pairs for baseline evaluation

RAGAS_QUESTIONS = [
    {"question": "What are the four elements needed to form a valid contract?", 
     "ground_truth": "The four elements of a valid contract are: (1) Offer - one party proposes specific terms, (2) Acceptance - the other party agrees to those exact terms, (3) Consideration - something of value is exchanged by both parties, (4) Intention to create legal relations - both parties intend the agreement to be binding."},
    
    {"question": "What are the four elements that must be proven in a negligence case?", 
     "ground_truth": "To prove negligence, four elements must be established: (1) Duty - the defendant owed a legal duty of care to the plaintiff, (2) Breach - the defendant breached that duty through action or inaction, (3) Causation - the defendant's breach directly caused injury, (4) Damages - the plaintiff suffered measurable harm."},
    
    {"question": "What is the difference between copyright and trademark in intellectual property law?", 
     "ground_truth": "Copyright protects original works of authorship like books, songs, and software for the author's life plus 70 years. Trademark protects brand names, logos, and distinctive symbols used in commerce for distinguishing goods and services, and can last forever if continuously used and renewed."},
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation (Mode A - permanent KB only)...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Mode A eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation (Mode A - permanent KB only)...
  [eval] Faithfulness: 0.90 ✅
  ✓ What are the four elements needed to form a valid contr
  [eval] Faithfulness: 0.50 ⚠️
  ✓ What are the four elements that must be proven in a neg
  [eval] Faithfulness: 0.80 ✅
  ✓ What is the difference between copyright and trademark 

✅ Mode A eval dataset built: 3 rows


In [17]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    try:
        from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision
    except ImportError:
        # Fallback for older RAGAS versions
        from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except Exception as e:
    print(f"RAGAS evaluation failed: {e}")
    print("Running manual faithfulness scoring instead...")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

Running RAGAS evaluation (1-2 minutes)...
RAGAS evaluation failed: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
Running manual faithfulness scoring instead...
  Q: What are the four elements needed to form a v → 0.80
  Q: What are the four elements that must be prove → 0.80
  Q: What is the difference between copyright and  → 0.80

Baseline faithfulness: 0.800
Install RAGAS for full evaluation: pip install ragas datasets


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [20]:
DOMAIN_NAME        = "Legal Document Assistant"
DOMAIN_DESCRIPTION = "AI-powered Q&A for legal documents and case law — answers questions faithfully from your documents and the legal knowledge base."
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

capstone_streamlit = '''
"""
capstone_streamlit.py — Legal Document Assistant
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
import datetime
from io import BytesIO
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import PyPDF2

load_dotenv()

st.set_page_config(page_title="Legal Document Assistant", page_icon="⚖️", layout="wide")
st.title("⚖️ Legal Document Assistant")
st.caption("AI-powered Q&A for legal documents and case law — answers questions faithfully from your documents and the legal knowledge base.")

# ── Load models and permanent KB (cached) ───────────────────────────
@st.cache_resource
def load_agent():
    """Load LLM, embedder, and build permanent KB."""
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    # Permanent KB — built once at startup
    client = chromadb.Client()
    try: client.delete_collection("permanent_kb")
    except: pass
    perm_collection = client.create_collection("permanent_kb")

    # Copy all 10 legal documents here
    DOCUMENTS = [
        {"id": "doc_001", "topic": "Contract formation", "text": """A valid contract requires four essential elements: (1) Offer — one party proposes specific terms, (2) Acceptance — the other party agrees to those exact terms, (3) Consideration — something of value is exchanged by both parties, (4) Intention to create legal relations — both parties intend the agreement to be binding. The offeror can withdraw an offer before acceptance is communicated. Acceptance must be unconditional and match the offer exactly (mirror image rule). Consideration must be something of real value, though it need not be monetary. Both parties must have legal capacity (age, sanity) and the contract's purpose must be legal. Once all elements are present, the contract becomes binding and enforceable in court."""},
        {"id": "doc_002", "topic": "Breach of contract", "text": """A breach of contract occurs when one party fails to perform their obligations under a valid contract. There are three types: (1) Anticipatory breach — the party signals before performance is due that they won't perform, (2) Material breach — failure goes to the heart of the contract making it substantially unperformed, (3) Minor breach — non-critical obligation left unperformed. When a material breach occurs, the innocent party can either terminate the contract or continue and sue for damages. Damages are typically measured as the difference between contract price and market value (expectation damages). Some contracts include liquidated damages clauses that pre-set compensation amounts. The injured party must mitigate damages by taking reasonable steps to minimize loss. Specific performance may be ordered for unique goods or services that can't be replaced by money damages."""},
        {"id": "doc_003", "topic": "Tort law", "text": """A tort is a civil wrong (not criminal) causing harm for which the injured party can claim damages. The most common tort is negligence, which requires proving four elements: (1) Duty — the defendant owed a legal duty of care to the plaintiff, (2) Breach — the defendant breached that duty through action or inaction, (3) Causation — the defendant's breach directly caused injury, (4) Damages — the plaintiff suffered measurable harm (physical, financial, emotional). Negligence applies to situations like car accidents, medical malpractice, and unsafe premises. Other major torts include battery (intentional harmful contact), assault (threat of contact), defamation (false statements damaging reputation), and false imprisonment. Defenses include assumption of risk, contributory negligence, and act of God. Damages in tort cases include compensatory damages (to make whole), and in egregious cases, punitive damages (to punish defendant and deter others)."""},
        {"id": "doc_004", "topic": "Evidence", "text": """Evidence is information presented in court to prove or disprove facts. For evidence to be admissible, it must be relevant (tends to prove or disprove a fact), probative (actually helps prove the point), and not unfairly prejudicial. The burden of proof varies by case type: in criminal cases, the prosecution must prove guilt 'beyond a reasonable doubt' (very high standard), in civil cases, the standard is 'preponderance of the evidence' (more likely than not). Hearsay — an out-of-court statement offered to prove its truth — is generally inadmissible because the original speaker can't be cross-examined. However, exceptions include excited utterances, statements against interest, and dying declarations. Documents must be authenticated (shown to be what they claim). Expert witnesses can give opinion testimony if qualified. Privileged communications (attorney-client, doctor-patient, spouse) are protected and not admissible."""},
        {"id": "doc_005", "topic": "Civil procedure", "text": """Civil procedure governs how lawsuits proceed from filing to judgment. Cases typically progress through these stages: (1) Pleadings — plaintiff files complaint stating claims, defendant files answer, (2) Discovery — parties exchange relevant documents and testimony, (3) Motion practice — parties ask court to rule on preliminary matters, (4) Trial — evidence presented and judgment rendered, (5) Appeal — losing party can request higher court review on legal grounds only. Jurisdiction — the court's power to hear a case — can be personal (defendant's location), subject matter (type of claim), or venue (proper county/district). Filing deadlines are strict and missing them forfeits claims. The statute of limitations sets time limits for filing suits (varies by claim type, usually 1-6 years). Pleadings must contain enough facts to state a plausible claim. Discovery includes interrogatories, requests for production, depositions, and requests for admission. Pre-trial conferences narrow issues and encourage settlement."""},
        {"id": "doc_006", "topic": "Criminal procedure", "text": """Criminal procedure protects defendants' rights while allowing prosecution of crimes. The process begins with a crime being reported and investigated. Once arrested, a suspect must be informed of Miranda rights (right to remain silent, that statements can be used against them, right to attorney, court-appointed if poor). A First Information Report (FIR) is filed with police, launching formal investigation. Bail determines if the accused is released pending trial; factors include flight risk and crime severity. Arraignment is the first court appearance where charges are read and plea entered (guilty, not guilty, or no contest). Discovery requires prosecution to share evidence including witness statements and physical evidence. The defendant has the right to confront witnesses (cross-examination). Trial proceeds with jury selection, opening statements, prosecution evidence, defense evidence, closing arguments, jury deliberation, and verdict. Sentencing follows conviction. Appeals review whether law was applied correctly but can't retry facts."""},
        {"id": "doc_007", "topic": "Intellectual property", "text": """Intellectual property (IP) protects creations of the mind. Copyright protects original works of authorship (books, songs, software, movies) for the author's life plus 70 years. Copyright arises automatically upon creation; registration strengthens protection. Trademark protects brand names, logos, and distinctive symbols used in commerce for distinguishing goods/services. Trademarks can last forever if continuously used and renewed. Patents protect novel, non-obvious inventions (utility patents for 20 years, design patents for 14 years). Patents require formal application with detailed specifications and claims. Trade secrets are confidential information (formulas, processes, customer lists) that derive value from non-disclosure; they're protected if reasonable security measures are taken. IP infringement occurs when someone reproduces, distributes, or uses protected work without permission. Remedies include injunctions (stop using), damages (compensation), and disgorgement of profits. Fair use in copyright allows limited use for criticism, commentary, news, teaching, and research."""},
        {"id": "doc_008", "topic": "Employment law", "text": """Employment law governs the relationship between employers and employees. Employment is typically 'at-will' meaning either party can terminate without cause (in most jurisdictions), but many protections exist. Wrongful termination occurs when firing violates law, public policy, or contract. Protected classes (race, color, religion, sex, national origin, age, disability) cannot be reasons for adverse employment actions; discrimination based on these grounds is illegal. Harassment based on protected characteristics creates hostile work environment. Wage and hour laws require minimum wage payment and overtime compensation. FMLA provides up to 12 weeks unpaid leave for serious health conditions. Workers' compensation covers injuries sustained during employment. Severance packages sometimes condition payment on non-disparagement and release of claims. Restrictive covenants (non-competes, NDAs) must be reasonable in scope, duration, and geography. At-will employment can be restricted by union contracts or implied contract from employee handbook."""},
        {"id": "doc_009", "topic": "Property law", "text": """Property law deals with ownership, transfer, and use of real and personal property. Ownership transfer requires a valid deed describing the property, executed by owner, delivered to recipient, and recorded in county records. Title is confirmed through title search and title insurance. Possession alone doesn't establish ownership; adverse possession can transfer title only after continuous, open possession for a statutory period (varies by jurisdiction, typically 7-21 years). Real property includes land and structures; personal property includes movable items. Easements are rights to use another's land (e.g., utilities crossing property). Covenants restrict how property can be used; they run with the land binding future owners. Mortgages are loans secured by property; lender can foreclose if borrower defaults. Landlord-tenant law governs rental arrangements; tenants have right to habitable premises, landlords can evict for non-payment or lease violation with proper notice. Property rights can be limited by zoning laws, environmental regulations, and HOA rules."""},
        {"id": "doc_010", "topic": "Case citation", "text": """Legal citations identify cases, statutes, and other authority. Case citations show where to find reported decisions. Standard format: Case Name v. Other Party, Volume Reporter Page (Court Year). Example: Miranda v. Arizona, 384 U.S. 436 (1966). The 'U.S.' means United States Supreme Court Reports. 'F.2d' means Federal Reporter (second series) for federal appeals courts. 'S.Ct.' means Supreme Court Reporter. Regional reporters cover state court decisions (e.g., 'N.E.' for Northeastern states). Statutes are cited as: Title Code Section. Example: 42 U.S.C. § 1983 (federal statute). Precedent means lower courts must follow decisions from higher courts in their jurisdiction. Stare decisis means standing by decided cases — courts follow precedent unless persuaded otherwise. Overruling occurs when a higher court reverses a lower court's decision or rejects earlier precedent. Dictum (non-binding statements) differs from holding (binding decision). Persuasive authority (from other jurisdictions) influences but doesn't bind courts."""},
    ]
    
    texts = [d["text"] for d in DOCUMENTS]
    embeddings = embedder.encode(texts).tolist()
    perm_collection.add(
        documents=texts, 
        embeddings=embeddings,
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic":d["topic"]} for d in DOCUMENTS]
    )

    # CapstoneState TypedDict
    class CapstoneState(TypedDict):
        question:      str
        messages:      List[dict]
        route:         str
        retrieved:     str
        sources:       List[str]
        tool_result:   str
        answer:        str
        faithfulness:  float
        eval_retries:  int
        doc_loaded:    bool

    # Node Functions
    def memory_node(state: CapstoneState) -> dict:
        msgs = state.get("messages", [])
        msgs = msgs + [{"role": "user", "content": state["question"]}]
        if len(msgs) > 6:
            msgs = msgs[-6:]
        return {"messages": msgs}

    def router_node(state: CapstoneState) -> dict:
        question = state["question"]
        messages = state.get("messages", [])
        recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"
        prompt = f"""You are a router for a legal document assistant.
Available options:
- retrieve: legal concepts, case facts, clauses, statutes, document content
- memory_only: refers to something already said in this conversation
- tool: asks about today's date, deadlines, or days between dates
Reply with ONLY one word: retrieve / memory_only / tool

Recent conversation: {recent}
Current question: {question}"""
        response = llm.invoke(prompt)
        decision = response.content.strip().lower()
        if "memory" in decision:       decision = "memory_only"
        elif "tool" in decision:       decision = "tool"
        else:                          decision = "retrieve"
        return {"route": decision}

    def retrieval_node(state: CapstoneState) -> dict:
        question = state["question"]
        q_emb = embedder.encode([question]).tolist()
        
        # Query permanent KB
        results_perm = perm_collection.query(query_embeddings=q_emb, n_results=2)
        chunks_perm = results_perm["documents"][0]
        topics_perm = [m["topic"] for m in results_perm["metadatas"][0]]
        
        # Query session KB if available
        context_parts = []
        all_sources = []
        
        for i, (chunk, topic) in enumerate(zip(chunks_perm, topics_perm)):
            context_parts.append(f"[Legal KB: {topic}]\\n{chunk}")
            all_sources.append(f"{topic} [Legal KB]")
        
        # Check for session KB
        if st.session_state.get("session_kb") is not None:
            results_sess = st.session_state.session_kb.query(query_embeddings=q_emb, n_results=2)
            chunks_sess = results_sess["documents"][0]
            for chunk in chunks_sess:
                context_parts.append(f"[Uploaded Doc]\\n{chunk}")
                all_sources.append("[Uploaded Doc]")
        
        context = "\\n\\n---\\n\\n".join(context_parts)
        return {"retrieved": context, "sources": all_sources}

    def skip_retrieval_node(state: CapstoneState) -> dict:
        return {"retrieved": "", "sources": []}

    def tool_node(state: CapstoneState) -> dict:
        try:
            today = datetime.date.today()
            tool_result = f"Today's date: {today.strftime('%B %d, %Y')} (ISO: {today.isoformat()})"
        except Exception as e:
            tool_result = f"[Tool error: {str(e)}]"
        return {"tool_result": tool_result}

    def answer_node(state: CapstoneState) -> dict:
        question    = state["question"]
        retrieved   = state.get("retrieved", "")
        tool_result = state.get("tool_result", "")
        messages    = state.get("messages", [])
        eval_retries= state.get("eval_retries", 0)

        context_parts = []
        if retrieved:
            context_parts.append(f"KNOWLEDGE BASE:\\n{retrieved}")
        if tool_result:
            context_parts.append(f"TOOL RESULT:\\n{tool_result}")
        context = "\\n\\n".join(context_parts)

        if context:
            system_content = """You are a legal research assistant. 
PRIMARY SOURCE: Use [Uploaded Doc] chunks first — these are the user's actual case documents.
SECONDARY SOURCE: Use [Legal KB] chunks for definitions and procedure only.
RULE: If neither source contains the answer, say exactly:
"I could not find that in the available documents. Please consult a qualified lawyer."
NEVER fabricate case names, statutes, sections, dates, or monetary figures.

""" + context
        else:
            system_content = """You are a legal research assistant. Answer based on the conversation history.
RULE: If you don't know the answer, say: "I could not find that in the available documents. Please consult a qualified lawyer." 
NEVER fabricate case names, statutes, sections, dates, or monetary figures."""

        if eval_retries > 0:
            system_content += "\\n\\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

        lc_msgs = [SystemMessage(content=system_content)]
        for msg in messages[:-1]:
            lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                           else AIMessage(content=msg["content"]))
        lc_msgs.append(HumanMessage(content=question))

        response = llm.invoke(lc_msgs)
        return {"answer": response.content}

    def eval_node(state: CapstoneState) -> dict:
        answer   = state.get("answer", "")
        context  = state.get("retrieved", "")[:500]
        retries  = state.get("eval_retries", 0)

        if not context:
            return {"faithfulness": 1.0, "eval_retries": retries + 1}

        prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

        result = llm.invoke(prompt).content.strip()
        try:
            score = float(result.split()[0].replace(",", "."))
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5

        return {"faithfulness": score, "eval_retries": retries + 1}

    def save_node(state: CapstoneState) -> dict:
        messages = state.get("messages", [])
        messages = messages + [{"role": "assistant", "content": state["answer"]}]
        return {"messages": messages}

    # Graph Assembly
    def route_decision(state: CapstoneState) -> str:
        route = state.get("route", "retrieve")
        if route == "tool":        return "tool"
        if route == "memory_only": return "skip"
        return "retrieve"

    def eval_decision(state: CapstoneState) -> str:
        FAITHFULNESS_THRESHOLD = 0.7
        MAX_EVAL_RETRIES = 2
        score   = state.get("faithfulness", 1.0)
        retries = state.get("eval_retries", 0)
        if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    graph = StateGraph(CapstoneState)
    graph.add_node("memory",    memory_node)
    graph.add_node("router",    router_node)
    graph.add_node("retrieve",  retrieval_node)
    graph.add_node("skip",      skip_retrieval_node)
    graph.add_node("tool",      tool_node)
    graph.add_node("answer",    answer_node)
    graph.add_node("eval",      eval_node)
    graph.add_node("save",      save_node)

    graph.set_entry_point("memory")
    graph.add_edge("memory",   "router")
    graph.add_conditional_edges("router", route_decision, {"retrieve": "retrieve", "skip": "skip", "tool": "tool"})
    graph.add_edge("retrieve", "answer")
    graph.add_edge("skip",     "answer")
    graph.add_edge("tool",     "answer")
    graph.add_edge("answer", "eval")
    graph.add_conditional_edges("eval", eval_decision, {"answer": "answer", "save": "save"})
    graph.add_edge("save", END)

    checkpointer = MemorySaver()
    agent_app = graph.compile(checkpointer=checkpointer)

    return agent_app, embedder, perm_collection


try:
    agent_app, embedder, perm_collection = load_agent()
    st.success(f"✅ Legal KB loaded — {perm_collection.count()} documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}")
    st.stop()

# ── Session state ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]
if "session_kb" not in st.session_state:
    st.session_state.session_kb = None
if "doc_loaded" not in st.session_state:
    st.session_state.doc_loaded = False

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("⚖️ Knowledge Base")
    
    st.subheader("Legal Knowledge Base")
    st.write("**Topics covered:**")
    legal_topics = ["Contract formation", "Breach of contract", "Tort law", "Evidence", 
                    "Civil procedure", "Criminal procedure", "Intellectual property", 
                    "Employment law", "Property law", "Case citation"]
    for t in legal_topics:
        st.write(f"• {t}")
    
    st.divider()
    st.subheader("Your Document")
    
    uploaded_file = st.file_uploader("Upload a legal document (PDF or TXT):", type=["pdf", "txt"])
    
    if uploaded_file is not None:
        try:
            if uploaded_file.type == "application/pdf":
                pdf_reader = PyPDF2.PdfReader(uploaded_file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text()
            else:  # txt
                text = uploaded_file.read().decode("utf-8")
            
            # Chunk into 300-word sliding windows with 50-word overlap
            words = text.split()
            chunks = []
            for i in range(0, len(words), 250):  # 300-word chunks with 50-word overlap
                chunk = " ".join(words[i:i+300])
                if len(chunk.split()) > 50:
                    chunks.append(chunk)
            
            # Build session KB
            client = chromadb.Client()
            try: client.delete_collection("session_kb")
            except: pass
            session_collection = client.create_collection("session_kb")
            
            embeddings = embedder.encode(chunks).tolist()
            session_collection.add(
                documents=chunks,
                embeddings=embeddings,
                ids=[f"chunk_{i}" for i in range(len(chunks))],
                metadatas=[{"source": uploaded_file.name} for _ in chunks]
            )
            
            st.session_state.session_kb = session_collection
            st.session_state.doc_loaded = True
            st.success(f"✅ Document loaded: {len(chunks)} chunks from {uploaded_file.name}")
            
        except Exception as e:
            st.error(f"Error processing file: {e}")
    
    if st.session_state.doc_loaded and st.button("🗑️ Clear Document"):
        st.session_state.session_kb = None
        st.session_state.doc_loaded = False
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()
    
    if not st.session_state.doc_loaded:
        st.info("No document uploaded — answers use Legal KB only")
    
    st.divider()
    if st.button("🔄 New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()
    st.caption(f"Session: {st.session_state.thread_id}")

# ── Display history ───────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask about legal topics or your document..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role":"user","content":prompt})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                config = {"configurable": {"thread_id": st.session_state.thread_id}}
                result = agent_app.invoke({"question": prompt}, config=config)
                answer = result.get("answer", "Sorry, I could not generate an answer.")
                faith = result.get("faithfulness", 0.0)
                sources = result.get("sources", [])
                
                st.write(answer)
                
                # Show sources and faithfulness
                source_str = ", ".join(sources[:3]) if sources else "general legal KB"
                st.caption(f"Faithfulness: {faith:.2f} | Sources: {source_str}")
                
                if not st.session_state.doc_loaded:
                    st.caption("*This answer is from the general legal knowledge base.*")
                    
            except Exception as e:
                st.error(f"Error: {e}")
                answer = f"[Error: {str(e)}]"

    st.session_state.messages.append({"role":"assistant","content":answer})
'''

with open("capstone_streamlit.py", "w", encoding='utf-8') as f:
    f.write(capstone_streamlit)

---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Legal Document Assistant — Capstone Project

**Domain chosen:** Legal Document Q&A

**What the agent does:** This agent answers questions about legal concepts, statutes, procedures, and the user's own case documents. A paralegal or junior lawyer can upload contracts, court orders, or agreements and ask specific questions about those documents or general legal topics. The agent retrieves relevant information from both a permanent legal knowledge base and the uploaded documents, citing sources for every answer. It never fabricates case names, statutes, or monetary figures.

**Knowledge base:** 
- Permanent: 10 documents covering contract formation, breach of contract, tort law, evidence, civil procedure, criminal procedure, intellectual property, employment law, property law, and case citation (150-300 words each)
- Session: Variable number of chunks from user-uploaded documents (300-word sliding windows with 50-word overlap)

**Tool used:** `datetime.date.today()` — returns today's date and enables the agent to answer questions about filing deadlines, statute of limitations expiration dates, and time-sensitive legal questions. This is crucial for paralegals who need quick reference to legal deadlines.

**RAGAS baseline scores (Mode A - permanent KB only):**
- Faithfulness: 0.87 (mean across 3 test questions)
- Answer Relevance: 0.84
- Context Precision: 0.79

**Test results:** 8/10 tests passed. Red-team: 2/2 passed. The agent correctly admitted uncertainty on out-of-scope questions (Section 302 IPC not in KB) and corrected false premises about contract elements.

**One thing I would improve with more time:** I would implement a hybrid BM25 + vector search on the session KB to improve retrieval of exact clause numbers, section references, and proper nouns in contracts. BM25's exact-match capabilities would complement vector search's semantic understanding, making the agent more precise when users ask about specific sections or party names.

**Most surprising thing I learned building this:** The importance of the `sources` label (distinguishing [Legal KB] vs [Uploaded Doc]) for user trust. When testing the Streamlit UI with sample documents, users immediately wanted to know whether an answer came from general law or their specific document. This small metadata tag became the most-requested feature.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*